<small>Dataset: 8-puzzle states (defined in code)</small>

# Q08. Eight Puzzle A Star

Move the blank tile to transform an initial board into the goal board.


## Question from Classroom

Solve the 8-Puzzle Using A* Search 
Problem statement: Implement A* Search to transform a given 8-puzzle configuration into a 
goal configuration. 
Tasks to be performed: 
1. Define the initial and goal states. 
2. Generate valid moves of the empty tile. 
3. Use Manhattan distance as the heuristic. 
4. Implement A* Search with repeated-state detection. 
5. Display the sequence of puzzle states from start to goal. 
6. Report the number of moves and states explored.


## Understand before coding

1. Store all 9 tiles as a tuple; 0 is blank.
2. Generate neighbouring states by swapping the blank.
3. Sum Manhattan distances of numbered tiles.
4. Use A* with best-cost repeated-state detection.
5. Print each board on the solution path.


## Define A* and the puzzle solver

The common search function is included in this notebook, so no other notebook is needed.


**Code layout:** 6 listed tasks → 6 code blocks. Run them in order. Imports and helper definitions are included in the relevant task. Where the PDF repeats a task number, these blocks follow its listed order. Learned preprocessing is fitted only after splitting.


## Task 1. Define the initial and goal states.

This block implements the task above. Variables from earlier blocks are reused.


In [1]:
import heapq
start=(1,2,3,4,0,6,7,5,8)
goal=(1,2,3,4,5,6,7,8,0)
if len(start)!=9 or set(start)!=set(range(9)) or len(goal)!=9 or set(goal)!=set(range(9)):
    raise ValueError("States must each contain 0..8 exactly once")
print("Start:",start,"Goal:",goal)


Start: (1, 2, 3, 4, 0, 6, 7, 5, 8) Goal: (1, 2, 3, 4, 5, 6, 7, 8, 0)


## Task 2. Generate valid moves of the empty tile.

This block implements the task above. Variables from earlier blocks are reused.


In [2]:
def neighbours(state):
    blank=state.index(0)
    row,col=divmod(blank,3)
    states=[]
    for r,c in [(row+1,col),(row-1,col),(row,col+1),(row,col-1)]:
        if 0<=r<3 and 0<=c<3:
            other=r*3+c
            next_state=list(state)
            next_state[blank],next_state[other]=next_state[other],next_state[blank]
            states.append(tuple(next_state))
    return states
print("Valid next states:",neighbours(start))


Valid next states: [(1, 2, 3, 4, 5, 6, 7, 0, 8), (1, 0, 3, 4, 2, 6, 7, 5, 8), (1, 2, 3, 4, 6, 0, 7, 5, 8), (1, 2, 3, 0, 4, 6, 7, 5, 8)]


## Task 3. Use Manhattan distance as the heuristic.

This block implements the task above. Variables from earlier blocks are reused.


In [3]:
goal_positions={tile:divmod(index,3) for index,tile in enumerate(goal)}
def h(state):
    estimate=0
    for index,tile in enumerate(state):
        if tile!=0:
            row,col=divmod(index,3)
            target_row,target_col=goal_positions[tile]
            estimate+=abs(row-target_row)+abs(col-target_col)
    return estimate
print("Initial Manhattan estimate:",h(start))


Initial Manhattan estimate: 2


## Task 4. Implement A* Search with repeated-state detection.

This block implements the task above. Variables from earlier blocks are reused.


In [4]:
rank={tile:index for index,tile in enumerate(goal) if tile!=0}
sequence=[rank[tile] for tile in start if tile!=0]
inversions=sum(sequence[i]>sequence[j] for i in range(8) for j in range(i+1,8))
queue=[(h(start),0,start,[start])] if inversions%2==0 else []
best_cost={start:0}
path=[]
explored=0
while queue:
    estimate,cost,state,route=heapq.heappop(queue)
    if cost!=best_cost[state]:continue
    explored+=1
    if state==goal:path=route;break
    for next_state in neighbours(state):
        new_cost=cost+1
        if new_cost<best_cost.get(next_state,float("inf")):
            best_cost[next_state]=new_cost
            heapq.heappush(queue,(new_cost+h(next_state),new_cost,next_state,route+[next_state]))


## Task 5. Display the sequence of puzzle states from start to goal.

This block implements the task above. Variables from earlier blocks are reused.


In [5]:
for step,state in enumerate(path):
    print("Step",step)
    for index in [0,3,6]:print(*state[index:index+3])
    print()
if not path:print("Unsolvable puzzle")


Step 0
1 2 3
4 0 6
7 5 8

Step 1
1 2 3
4 5 6
7 0 8

Step 2
1 2 3
4 5 6
7 8 0



## Task 6. Report the number of moves and states explored.

This block implements the task above. Variables from earlier blocks are reused.


In [6]:
print("Number of moves:",len(path)-1 if path else "No solution")
print("States explored:",explored)


Number of moves: 2
States explored: 3


## Viva preparation guide

**1. How should I explain my program?**

State the input/state, allowed moves or constraints, the algorithm, stopping condition and printed result. Then trace one small example.

**2. How should I test it?**

Try the supplied example, a trivial solved case and an impossible or invalid case. Check the actual route/board/constraints, not just the final success message.

**3. What is the state representation?**

A tuple of 9 tiles in row-major order; 0 is blank. Tuples are hashable and can be dictionary keys.

**4. How are successors generated?**

Find the blank, consider its in-bounds four-direction neighbours, and swap 0 with the chosen neighbour.

**5. What is the heuristic?**

For each numbered tile, add its row and column distance from its goal location. Do not include the blank.

**6. Why is this heuristic admissible?**

Each legal move changes the position of one numbered tile by one step, so the summed Manhattan distances cannot exceed the moves required.

**7. Why repeated-state detection?**

Moving the blank back and forth can create cycles. best-cost checks prevent repeatedly exploring more expensive versions of the same board.

**8. Why is a solvability check useful?**

Some permutations cannot reach the goal. For a 3x3 board, parity of inversions relative to the goal detects them before search.

**9. What is an inversion?**

A pair of numbered tiles in the wrong relative order, ignoring zero. The code ranks tiles by the supplied goal, so it supports nonstandard goals too.

**10. What is the demo output?**

A sequence of 3 boards representing 2 moves. An already solved initial board requires 0 moves.

**11. How many states can exist?**

There are 9! permutations, split into two equal reachable parity classes of 9!/2 states. This bounds the reachable state set for a valid fixed goal.

**12. Complexity and memory?**

A* can use substantial time and memory because it retains frontier and best-cost data. General search can grow exponentially with solution depth; this teaching code also copies paths.


### Task-to-code map

- **Block 1:** Define the initial and goal states.
- **Block 2:** Generate valid moves of the empty tile.
- **Block 3:** Use Manhattan distance as the heuristic.
- **Block 4:** Implement A* Search with repeated-state detection.
- **Block 5:** Display the sequence of puzzle states from start to goal.
- **Block 6:** Report the number of moves and states explored.
